# Topic 11 — Solutions: Exploratory Analysis & Visualization

*Reference solutions for the objective tasks. Try the practice first.* The Warm-Up concept questions, Interview Lens and Reflection have **no key** — by design.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 30)
RAW = '../datasets/raw/'
products = pd.read_csv(RAW+'products.csv', dtype={'product_id':str})
products['list_price'] = np.where(products['list_price']<0, np.nan, products['list_price'])
orders = pd.read_csv(RAW+'orders.csv', dtype={'order_id':str,'customer_id':str})
orders['channel'] = orders['channel'].str.strip().str.title()
orders['status'] = orders['status'].str.strip().str.lower()
orders['order_date'] = pd.to_datetime(orders['order_date'], format='mixed', dayfirst=True, errors='coerce')
orders['month'] = orders['order_date'].dt.to_period('M').astype(str)
items = pd.read_csv(RAW+'order_items.csv', dtype={'order_id':str,'product_id':str})
items['line_revenue'] = items['quantity']*items['unit_price']*(1-items['discount'])
master = (items.merge(products[['product_id','category','unit_cost']], on='product_id', how='left')
              .merge(orders[['order_id','channel','status','month','order_date']], on='order_id', how='left'))
master['line_cogs'] = master['quantity']*master['unit_cost']
master['line_profit'] = master['line_revenue'] - master['line_cogs']
oi = master  # alias
ts = master.dropna(subset=['order_date']).set_index('order_date').sort_index()
monthly = ts['line_revenue'].resample('ME').sum()


### Warm-Up — NumPy recall (self-check)

In [ ]:
data = np.array([1,1,2,3,3,3,4])
counts, edges = np.histogram(data, bins=4)
print(counts)

### Core lesson tasks

In [ ]:
ax = monthly.plot(figsize=(10,4)); monthly.rolling(3).mean().plot(ax=ax)
ax.set_title('Aurora revenue is seasonal, not declining'); ax.set_ylabel('Revenue (£)'); plt.show()

In [ ]:
channel_rev = master.groupby('channel')['line_revenue'].sum().sort_values(ascending=False)
ax = channel_rev.plot(kind='bar', title='Online dominates revenue; Phone is marginal'); ax.set_ylabel('Revenue (£)'); plt.tight_layout(); plt.show()

In [ ]:
order_rev = master.groupby('order_id')['line_revenue'].sum()
order_rev.plot(kind='hist', bins=50, title='Per-order revenue is right-skewed'); plt.xlabel('Order revenue (£)'); plt.show()

### Mixed review (earlier topics)

In [ ]:
returns = pd.read_csv(RAW+'returns.csv', dtype={'order_id':str})
rmonth = returns.merge(orders[['order_id','order_date']], on='order_id', how='left').dropna(subset=['order_date'])
print(rmonth.set_index('order_date')['refund_amount'].resample('ME').sum().tail())

In [ ]:
cat_profit = master.groupby('category')['line_profit'].sum().sort_values()
cat_profit.plot(kind='barh', title='Profit by category'); plt.show()

### Data detective

In [ ]:
print('Dashboard = 4 panels: revenue trend, revenue by channel, profit by category, profit by channel.')

### Interview Lens — discussion points (not full answers)
- **Q30:** reproduce from raw with a documented filter/join chain, check grain (row counts), show coverage of excluded rows, reconcile order-level vs line-level.
- **Q31:** one parameterised notebook, pinned versions, deterministic load, asserts on invariants, no manual spreadsheet steps.